# CMPE 401 Project 1 — VisDrone YOLOv11 (Colab)

Runs the **Part I baseline** (YOLOv11s, 640px) on a free Colab GPU, and serves as the base for the VRAM-heavy Part III/V runs (larger models, `imgsz=1280`) that don't fit on the RTX 3060 laptop. It clones the repo, installs deps, downloads + converts VisDrone, then trains/evaluates with the same scripts used locally.

**Outputs are written straight to Google Drive** (`MyDrive/cmpe401_project1/runs/`) so a Colab disconnect never loses your run — no manual save-at-the-end step.

**Steps:**
1. **Set the runtime to a GPU:** Runtime → Change runtime type → GPU (T4). Cell 2 should then print `CUDA available: True`.
2. Run cells 1→3 in order. Cell 3 should finish with ~457k boxes kept (same as local).
3. Run cell 4 to train (~2.5–4 h). It mounts Drive and writes weights/`results.csv`/plots there live. **Keep this tab open and active** — free Colab disconnects after ~90 min idle. If it dies mid-run, re-run cell 4 with `--resume` appended to continue from the last checkpoint.
4. Run cells 5–7 (no GPU required — they read from Drive). Cell 7 just confirms the files are in Drive.

> **Note:** after a long run, free Colab may rate-limit GPU access for several hours. If you can't get a GPU, wait for the reset, use Colab Pro, or run locally.

In [ ]:
# 1. Clone your repo (replace with your GitHub URL after you create it)
REPO_URL = "https://github.com/eijikuinbii/CMPE401-YOLOv11"
!git clone $REPO_URL project
%cd project

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Download + convert VisDrone-DET
!python scripts/download_visdrone.py --root datasets/VisDrone
!python scripts/convert_visdrone.py --root datasets/VisDrone

In [ ]:
# 4. Train — Part I baseline (YOLOv11s, 640px), writing outputs STRAIGHT TO GOOGLE DRIVE
# so they survive a disconnect even at the very end of the run.
#   project=<Drive path> : weights, results.csv and plots are written live into Drive
#   save_period=25       : also keep a resumable checkpoint every 25 epochs
# Image READS still come from the fast local disk (cell 3), so Drive output doesn't slow training.
# If training dies mid-run, re-run this SAME cell with --resume appended to continue.
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/cmpe401_project1/runs/baseline_s_640'   # reused by cells 5-7

!python scripts/train.py --model yolo11s.pt --imgsz 640 --batch 16 --epochs 100 \
    --name baseline_s_640 \
    --extra project=/content/drive/MyDrive/cmpe401_project1/runs save_period=25

In [ ]:
# 5. Evaluate on val and test-dev (reads best.pt from Drive; runs on GPU or CPU)
!python scripts/evaluate.py --weights {RUN_DIR}/weights/best.pt --split val --imgsz 640
!python scripts/evaluate.py --weights {RUN_DIR}/weights/best.pt --split test --imgsz 640

In [ ]:
# 6. Plots + summary table (pure CPU — just reads results.csv from Drive)
!python scripts/plot_curves.py {RUN_DIR}
!python scripts/summarize_runs.py {RUN_DIR}

In [ ]:
# 7. Confirm everything is already in Drive (cell 4 wrote outputs there live — no copy needed).
# These files persist after the session ends; download the run folder from Drive for the repo/report.
!ls -la {RUN_DIR} {RUN_DIR}/weights
print('\nAll outputs are in Drive at:', RUN_DIR)